# Drawing pixel baseline

Run cells from top to bottom. This notebook is the interactive entry point; reusable implementation and tested data/model logic remain in `src/train_drawing_models.py`. Set output paths before running the training cell to preserve earlier artifacts.

In [1]:
from pathlib import Path
import sys

# Find the repository root so package imports and relative artifact paths work from either launch location.
PROJECT_ROOT = Path.cwd()
# If Jupyter started inside notebooks/, move up one folder to the repository root.
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').is_dir(), 'Open this notebook from the repository or notebooks directory'
# Put the repository root on Python's import path for the src package.
sys.path.insert(0, str(PROJECT_ROOT))

from src.train_drawing_models import DEFAULT_IMAGE_ROOT, build_manifest, run_experiment

# Point to the local image dataset; override this if your data lives elsewhere.
IMAGE_ROOT = DEFAULT_IMAGE_ROOT
MODEL_PATH = PROJECT_ROOT / 'models' / 'finalization_baseline' / 'drawing_image_model.joblib'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'finalization_baseline' / 'drawing'
# Reuse the documented seed to make splits and estimators repeatable.
SEED = 42
# Preserve the baseline's 32 by 32 grayscale representation.
IMAGE_SIZE = 32
print('Image root:', IMAGE_ROOT)
print('Image root exists:', IMAGE_ROOT.exists())

Image root: C:\Users\daisy\Projects\AI\pd-ml-lab\data\Parkinson's drawings dataset
Image root exists: True


In [2]:
# Inspect discovered files, task/class counts, and provisional filename groups before training.
manifest = build_manifest(IMAGE_ROOT)
display(manifest.groupby(['task', 'label']).size().rename('images').to_frame())
display(manifest[['path', 'task', 'label', 'group']].head())
print('Rows:', len(manifest), '| candidate groups:', manifest['group'].nunique())

images
task   label            
spiral healthy        51
       parkinson      51
wave   healthy        51
       parkinson      51

,path,task,label,group
0,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V01
1,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V01
2,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V02
3,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V02
4,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V03


Rows: 204 | candidate groups: 28


In [3]:
# Run the full train/CV/selection/held-out evaluation and write the model and reports.
summary = run_experiment(
    image_root=IMAGE_ROOT,
    model_path=MODEL_PATH,
    results_dir=RESULTS_DIR,
    seed=SEED,
    image_size=IMAGE_SIZE,
)
summary

[spiral] extracting features from 102 images ...
[spiral] nested CV: handcrafted|logistic_regression
[spiral] nested CV: handcrafted|rbf_svm
[spiral] nested CV: handcrafted|random_forest
[spiral] nested CV: pixels_pca|logistic_regression
[spiral] nested CV: pixels_pca|rbf_svm
[spiral] nested CV: pixels_pca|random_forest
[spiral] nested CV: handcrafted+pixels_pca|logistic_regression
[spiral] nested CV: handcrafted+pixels_pca|rbf_svm
[spiral] nested CV: handcrafted+pixels_pca|random_forest
[wave] extracting features from 102 images ...
[wave] nested CV: handcrafted|logistic_regression
[wave] nested CV: handcrafted|rbf_svm
[wave] nested CV: handcrafted|random_forest
[wave] nested CV: pixels_pca|logistic_regression
[wave] nested CV: pixels_pca|rbf_svm
[wave] nested CV: pixels_pca|random_forest
[wave] nested CV: handcrafted+pixels_pca|logistic_regression
[wave] nested CV: handcrafted+pixels_pca|rbf_svm
[wave] nested CV: handcrafted+pixels_pca|random_forest

Out-of-fold results (chance ROC-A

{'study_scope': "Exploratory healthy-versus-known-Parkinson's image classification; not prospective early-risk prediction",
 'seed': 42,
 'image_size': 32,
 'label_mapping': {'healthy': 0, 'parkinson': 1},
 'group_key': 'class-scoped filename prefix V## (provisional participant token; verify with source documentation)',
 'input_root': "C:\\Users\\daisy\\Projects\\AI\\pd-ml-lab\\data\\Parkinson's drawings dataset",
 'n_images_after_exact_deduplication': 204,
 'n_unique_candidate_groups': 28,
 'evaluation': 'repeated (3x) nested StratifiedGroupKFold; inner CV tunes on training groups only; chance ROC-AUC = 0.5',
 'selection_bias_note': 'Best of 9 configurations per task is reported, so its estimate is mildly optimistic.',
 'per_task': {'spiral': {'n_images': 102,
   'n_groups': 28,
   'selected_configuration': 'handcrafted|logistic_regression',
   'selected_hyperparameters': {'classifier__C': 0.1},
   'cv_metrics_selected': {'outer_folds': 5,
    'image_roc_auc_mean': 0.7661155965654235,

In [4]:
# Confirm the expected artifacts were written.
print('Model:', MODEL_PATH, MODEL_PATH.exists())
print('Results:', RESULTS_DIR)
sorted(p.name for p in RESULTS_DIR.iterdir())

Model: c:\Users\daisy\Projects\AI\pd-ml-lab\models\finalization_baseline\drawing_image_model.joblib True
Results: c:\Users\daisy\Projects\AI\pd-ml-lab\results\finalization_baseline\drawing


['drawing_candidate_results.csv',
 'drawing_group_scores.csv',
 'drawing_training_metrics.json']